# ToaD Study — Complete Pipeline (v6)

One notebook. **Part 1** runs the experiments, **Part 2** interprets them. Everything persists to
Google Drive and every long loop checkpoints, so a dropped session resumes instead of restarting.v3 adds an **improved memory layout (ToaD+)** on top of the v2 baseline study. Two changes to howa model is stored, both applied to every arm:- **Succinct tree encoding.** ToaD indexes a *complete* binary tree, so an unbalanced tree pays for  phantom nodes. LightGBM grows leaf-wise, so its trees are very unbalanced: measured padding waste  runs 1.0x at depth 2 up to 13.5x at depth 8. ToaD+ replaces this with a LOUDS bitmap (2 bits per  real node, plus rank support) and rank-indexed payload arrays split by node type. **Lossless** —  predictions are bit-identical, only the accounting changes.- **Leaf-value quantization.** ToaD stores leaf values at 32 bits. ToaD+ stores them as b-bit fixed  point with a shared scale and offset. **Lossy** — must be checked against accuracy, which is why  it is a separate arm rather than folded into the layout.Note honestly in the paper: an improved layout lifts *all* arms including ToaD's own, so it does notby itself close the gap to their method. It is a separate contribution.Three changes carried over from the smoke test:1. **Leaf clustering arm.** Leaf values were 33% of the footprint versus 15% for thresholds, and the   ToaD paper's own conclusion flags better leaf reuse as open work. In trial runs this was by far   the strongest training-free lever.2. **Joint grid + iso-accuracy envelopes.** The smoke test varied one knob at a time, which cannot   be compared to their Figure 4. This version varies structure, ensemble size and `max_bin`   together, then takes the best model per memory budget, which is their protocol.3. **Vectorized predictor.** ~10x faster than the row-by-row walker, so Covertype is tractable.Reference: Herrmann, Stenkamp, Karic, Oehmcke, Gieseke. *Boosted Trees on a Diet.* ICLR 2026.arXiv:2510.26557. Repo: https://github.com/TinyAIoT/LightGBM-ToaD**Runtime:** roughly 15-25 minutes on Colab free tier with default settings. CPU only.**How to run:** Runtime -> Run all, then read the "Reading the results" section at the bottom.

## Step 1 — Install

In [ ]:
!pip install -q lightgbm scikit-learn catboost
print("done")

## Step 0 — Google Drive storageEverything this notebook produces goes to Drive: results, cached datasets, figures andcheckpoints. Three reasons that matters.- Colab wipes its local disk on restart. Results written locally disappear without warning.- Datasets are downloaded once and cached as `.npz`. Covertype in particular is slow to fetch.- The grid checkpoints after each dataset and seed pair. If the session drops mid-run, re-running  skips everything already finished instead of starting over. This is what makes `SEEDS = 1..12`  survivable.You will be asked to authorise Drive access the first time. Everything lands in`MyDrive/toad_project/`.

In [ ]:
import os

PROJ = "/content/drive/MyDrive/toad_project"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted")
except Exception as e:
    PROJ = "toad_project"          # not on Colab: fall back to a local folder
    print(f"not running on Colab ({type(e).__name__}); using local folder ./{PROJ}")

OUTDIR  = f"{PROJ}/results"
CACHE   = f"{PROJ}/cache"
FIGDIR  = f"{PROJ}/figures"
CKPTDIR = f"{PROJ}/checkpoints"
for _d in (OUTDIR, CACHE, FIGDIR, CKPTDIR):
    os.makedirs(_d, exist_ok=True)

print(f"\nresults     -> {OUTDIR}")
print(f"cache       -> {CACHE}")
print(f"figures     -> {FIGDIR}")
print(f"checkpoints -> {CKPTDIR}")

_ck = sorted(f for f in os.listdir(CKPTDIR) if f.endswith(".csv"))
if _ck:
    print(f"\n{len(_ck)} checkpoints already present - those will be skipped:")
    for f in _ck[:8]:
        print("   ", f)
    if len(_ck) > 8:
        print(f"    ... and {len(_ck)-8} more")
    print("\nDelete files in the checkpoints folder to force a re-run.")

## Step 2 — Imports and configuration`SEEDS` is the main dial. Leave it at `[1]` for a first pass to check everything runs, then set itto `list(range(1, 13))` for the real numbers. The ToaD authors used seeds 1-12.

In [ ]:
import copy, math, json, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
from sklearn.datasets import (load_breast_cancer, fetch_california_housing,
                              fetch_covtype, fetch_openml, load_diabetes)
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, r2_score, brier_score_loss

warnings.filterwarnings("ignore")

SEEDS = [1]                     # -> list(range(1, 13)) for the real run
MAX_TRAIN_ROWS = 30_000         # caps Covertype; raise if you have time
MAX_TEST_ROWS  = 20_000
# OUTDIR / CACHE / CKPTDIR come from Step 0

# joint grid
STRUCT   = [(2, 1), (4, 2), (8, 3), (16, 4), (32, 5)]   # (num_leaves, max_depth)
N_EST    = [10, 30, 60, 120]
MAX_BIN  = [255, 63, 15]
CB_DEPTH = [1, 2, 3, 4, 5]
CB_ITERS = [10, 30, 60, 120]

# post-hoc clustering configs, applied to the max_bin=255 models
CLUSTER_CFGS = ([("leaf", k) for k in (256, 64, 16, 8, 4)] +
                [("thr",  k) for k in (16, 8, 4)] +
                [("both", k) for k in (16, 8, 4)] +
                [("quant", b) for b in (16, 8, 4)])      # b-bit leaf quantization

print("lightgbm", lgb.__version__, "| seeds", SEEDS)

## Step 3 — The ToaD memory layout, as a scorerTheir Section 3.2 layout priced for any trained ensemble. Three assumptions I had to make, all ofwhich you must check against their repo before writing the paper, and all of which belong in yourmethods section:- **A1.** Metadata costs a flat 64 bits (4 fields x 16 bits).- **A2.** Node slots are fixed-width, sized by the widest case. Their Figure 3 hints at per-feature  variable widths, which would be slightly smaller. Fixed width never flatters your baselines.- **A3.** Leaf-vs-internal uses a reserved feature reference as sentinel, per their Section 4.2.

In [ ]:
def bits_for(n):
    n = max(int(n), 1)
    return int(math.ceil(math.log2(n))) if n > 1 else 0


def thr_width_and_type(values):
    # Smallest representation per their Section 3.2.1: 1 bit boolean, 2/4 bit small ints,
    # 8/16/32 bit for larger ints or floats. Returns (bit_width, "int" or "float").
    a = np.asarray(sorted(set(values)), dtype=np.float64)
    if a.size == 0:
        return 1, "int"
    if set(a.tolist()) <= {0.0, 1.0}:
        return 1, "int"
    if np.all(a == np.round(a)):
        lo, hi = a.min(), a.max()
        for w in (2, 4, 8, 16, 32):
            if lo >= 0 and hi <= 2 ** w - 1:
                return w, "int"
            if lo >= -(2 ** (w - 1)) and hi <= 2 ** (w - 1) - 1:
                return w, "int"
        return 32, "int"
    if np.array_equal(a.astype(np.float16).astype(np.float64), a):
        return 16, "float"
    return 32, "float"


def toad_bits(roots, n_input_features, meta_bits=64):
    feats, thr, leaves = collect(roots)
    nF, nL = len(feats), len(leaves)
    maxT = max((len(v) for v in thr.values()), default=1)
    fib = bits_for(n_input_features)

    global_thr_bits, map_bits = 0, 0
    for f in sorted(feats):
        w, _ = thr_width_and_type(thr[f])
        global_thr_bits += len(thr[f]) * w
        map_bits += fib + 3 + 1 + bits_for(maxT)

    leaf_array_bits = nL * 32
    node_bits = bits_for(nF + 1) + max(bits_for(maxT), bits_for(nL))
    trees_bits = sum((2 ** (tree_depth(r) + 1) - 1) * node_bits for r in roots)
    return meta_bits + global_thr_bits + map_bits + leaf_array_bits + trees_bits


def breakdown(roots, n_input_features, meta_bits=64):
    feats, thr, leaves = collect(roots)
    nF, nL = len(feats), len(leaves)
    maxT = max((len(v) for v in thr.values()), default=1)
    fib = bits_for(n_input_features)
    gt = sum(len(thr[f]) * thr_width_and_type(thr[f])[0] for f in feats)
    mp = nF * (fib + 3 + 1 + bits_for(maxT))
    lv = nL * 32
    nb = bits_for(nF + 1) + max(bits_for(maxT), bits_for(nL))
    tr = sum((2 ** (tree_depth(r) + 1) - 1) * nb for r in roots)
    return dict(metadata=meta_bits, thresholds=gt, feature_map=mp,
                leaf_values=lv, tree_arrays=tr)


def baseline_bits(roots, per_node=128):
    # Their Section 4.2 accounting: 128 bits/node float32, 64 bits/node fp16-quantized.
    return sum(count_nodes(r) for r in roots) * per_node


def array_layout_bits(roots, per_node=64):
    return sum(2 ** (tree_depth(r) + 1) - 1 for r in roots) * per_node


KB = lambda bits: bits / 8 / 1024


# ---------------- ToaD+ : the improved layout ----------------

def n_internal_leaves(root):
    ni = nl = 0
    st = [root]
    while st:
        n = st.pop()
        if is_leaf(n):
            nl += 1
        else:
            ni += 1
            st.append(n["left_child"]); st.append(n["right_child"])
    return ni, nl


def padding_waste(roots):
    # How many slots ToaD's complete-tree indexing pays for, per real node.
    act = sum(count_nodes(r) for r in roots)
    com = sum(2 ** (tree_depth(r) + 1) - 1 for r in roots)
    return com / act if act else np.nan


def toad_bits_plus(roots, n_input_features, meta_bits=64, leaf_bits=32,
                   rank_overhead=0.25, per_feature_thr_idx=True):
    # Succinct (LOUDS) structure + rank-indexed payloads + optional leaf quantization.
    # rank_overhead is the extra space for rank/select support; 0.25 is the usual budget.
    feats, thr, leaves = collect(roots)
    nF, nL = len(feats), len(leaves)
    maxT = max((len(v) for v in thr.values()), default=1)
    fib = bits_for(n_input_features)

    gt, mp = 0, 0
    for f in sorted(feats):
        w, _ = thr_width_and_type(thr[f])
        gt += len(thr[f]) * w
        mp += fib + 3 + 1 + bits_for(maxT)

    # quantized leaf table needs a scale and offset stored alongside
    leaf_arr = nL * leaf_bits + (2 * 32 if leaf_bits < 32 else 0)

    frb = bits_for(nF)        # no leaf sentinel needed: the bitmap says which nodes are leaves
    lib = bits_for(nL)

    struct_bits, pay_bits = 0, 0
    for r in roots:
        ni, nl = n_internal_leaves(r)
        struct_bits += int(np.ceil(2 * (ni + nl) * (1 + rank_overhead)))
        if per_feature_thr_idx:
            st = [r]
            while st:
                node = st.pop()
                if is_leaf(node):
                    pay_bits += lib
                else:
                    pay_bits += frb + bits_for(len(thr[node["split_feature"]]))
                    st.append(node["left_child"]); st.append(node["right_child"])
        else:
            pay_bits += ni * (frb + bits_for(maxT)) + nl * lib

    return meta_bits + gt + mp + leaf_arr + struct_bits + pay_bits


def quantize_leaves(roots, bits):
    # b-bit uniform fixed point over the leaf-value range. Lossy: always re-score accuracy.
    roots = copy.deepcopy(roots)
    _, _, lv = collect(roots)
    v = np.array(sorted(lv))
    lo, hi = v.min(), v.max()
    levels = 2 ** bits - 1
    for r in roots:
        st = [r]
        while st:
            n = st.pop()
            if is_leaf(n):
                x = float(n["leaf_value"])
                q = round((x - lo) / (hi - lo) * levels) if hi > lo else 0
                n["leaf_value"] = lo + q * (hi - lo) / levels
            else:
                st.append(n["left_child"]); st.append(n["right_child"])
    return roots

## Step 4 — Tree parsing and the vectorized predictor

In [ ]:
def is_leaf(node):
    return "leaf_value" in node and "split_feature" not in node


def tree_roots(booster):
    d = booster.dump_model()
    return [t["tree_structure"] for t in d["tree_info"]], d["max_feature_idx"] + 1


def tree_depth(node, d=0):
    if is_leaf(node):
        return d
    return max(tree_depth(node["left_child"], d + 1), tree_depth(node["right_child"], d + 1))


def count_nodes(node):
    if is_leaf(node):
        return 1
    return 1 + count_nodes(node["left_child"]) + count_nodes(node["right_child"])


def collect(roots):
    feats, thr, leaves = set(), {}, set()
    for r in roots:
        st = [r]
        while st:
            n = st.pop()
            if is_leaf(n):
                leaves.add(float(np.float32(n["leaf_value"])))
            else:
                f = n["split_feature"]
                feats.add(f)
                thr.setdefault(f, set()).add(float(n["threshold"]))
                st.append(n["left_child"]); st.append(n["right_child"])
    return feats, thr, leaves


def flatten_tree(root):
    # Dict tree -> flat arrays, so traversal can be done with numpy on all rows at once.
    feat, thr, left, right, val = [], [], [], [], []
    def add(node):
        i = len(feat)
        feat.append(-1); thr.append(0.0); left.append(-1); right.append(-1); val.append(0.0)
        if is_leaf(node):
            val[i] = float(node["leaf_value"]); return i
        feat[i] = int(node["split_feature"]); thr[i] = float(node["threshold"])
        l = add(node["left_child"]); r = add(node["right_child"])
        left[i] = l; right[i] = r
        return i
    add(root)
    return (np.array(feat, np.int64), np.array(thr, np.float64),
            np.array(left, np.int64), np.array(right, np.int64), np.array(val, np.float64))


def predict_vec(roots, X):
    # Raw scores. All rows walk the tree in lockstep, one numpy step per level.
    X = np.ascontiguousarray(np.asarray(X, np.float64))
    n = X.shape[0]
    out = np.zeros(n)
    for feat, thr, left, right, val in (flatten_tree(r) for r in roots):
        idx = np.zeros(n, np.int64)
        for _ in range(64):
            act = np.flatnonzero(feat[idx] >= 0)
            if act.size == 0:
                break
            ii = idx[act]
            go_left = X[act, feat[ii]] <= thr[ii]
            idx[act] = np.where(go_left, left[ii], right[ii])
        out += val[idx]
    return out

## Step 5 — Validation gateThe vectorized predictor must reproduce LightGBM's own raw scores exactly. If this fails, **stop** —every accuracy number for the clustering arms would be wrong.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
_m = lgb.LGBMClassifier(n_estimators=60, max_depth=4, num_leaves=16,
                        verbose=-1, random_state=1).fit(Xtr, ytr)
_r, _nf = tree_roots(_m.booster_)

err = np.abs(_m.booster_.predict(Xte, raw_score=True) - predict_vec(_r, Xte)).max()
print(f"vectorized predictor vs LightGBM, max abs error: {err:.2e}")
assert err < 1e-6, "Predictor mismatch. Stop and fix before going further."

b = breakdown(_r, _nf); tot = sum(b.values())
print("\nwhere the memory goes:")
for k, v in sorted(b.items(), key=lambda kv: -kv[1]):
    print(f"  {k:14s} {KB(v):7.3f} KB  {100*v/tot:5.1f}%")
print("\npadding waste under ToaD's complete-tree indexing (slots paid per real node):")
for nl, dep, ne in [(4, 2, 60), (16, 4, 60), (32, 5, 120), (64, 6, 120), (128, 8, 120)]:
    mm = lgb.LGBMClassifier(num_leaves=nl, max_depth=dep, n_estimators=ne,
                            verbose=-1, random_state=1).fit(Xtr, ytr)
    rr, _ = tree_roots(mm.booster_)
    print(f"  nl{nl}_d{dep}_n{ne:<4d}  {padding_waste(rr):5.2f}x   "
          f"ToaD {KB(toad_bits(rr, _nf)):8.3f} KB -> ToaD+ {KB(toad_bits_plus(rr, _nf)):8.3f} KB")

print("\nVALIDATION PASSED")

## Step 6 — Post-hoc compressionThree training-free levers, all applied to an already-fitted model:- `thr` — snap each feature's thresholds to `k` centres (the analogue of their threshold penalty)- `leaf` — snap all leaf values to `k` global centres (the axis their conclusion flags as open work)- `both` — apply leaf clustering at `k`, then threshold clustering at `k`Leaf clustering should behave better than threshold clustering, because leaf values are additivecontributions summed over many trees so small perturbations partly cancel, whereas a snappedthreshold reroutes a sample down an entirely wrong branch.

In [ ]:
def _kmeans_1d(v, k, iters=100):
    kk = min(k, len(v))
    if kk >= len(v):
        return v
    c = np.percentile(v, np.linspace(0, 100, kk))
    for _ in range(iters):
        a = np.abs(v[:, None] - c[None, :]).argmin(1)
        new = np.array([v[a == j].mean() if (a == j).any() else c[j] for j in range(kk)])
        if np.allclose(new, c):
            break
        c = new
    return c


def cluster_thresholds(roots, k):
    roots = copy.deepcopy(roots)
    _, thr, _ = collect(roots)
    mp = {}
    for f, vals in thr.items():
        v = np.array(sorted(vals))
        c = _kmeans_1d(v, k)
        mp[f] = {float(x): float(c[np.abs(c - x).argmin()]) for x in v}
    for r in roots:
        st = [r]
        while st:
            n = st.pop()
            if not is_leaf(n):
                n["threshold"] = mp[n["split_feature"]][float(n["threshold"])]
                st.append(n["left_child"]); st.append(n["right_child"])
    return roots


def cluster_leaves(roots, k):
    roots = copy.deepcopy(roots)
    _, _, leaves = collect(roots)
    v = np.array(sorted(leaves))
    c = _kmeans_1d(v, k)
    if len(c) >= len(v):
        return roots
    for r in roots:
        st = [r]
        while st:
            n = st.pop()
            if is_leaf(n):
                x = float(np.float32(n["leaf_value"]))
                n["leaf_value"] = float(c[np.abs(c - x).argmin()])
            else:
                st.append(n["left_child"]); st.append(n["right_child"])
    return roots


def apply_cluster(roots, kind, k):
    if kind == "quant":
        return quantize_leaves(roots, k)
    if kind == "thr":
        return cluster_thresholds(roots, k)
    if kind == "leaf":
        return cluster_leaves(roots, k)
    if kind == "both":
        return cluster_thresholds(cluster_leaves(roots, k), k)
    raise ValueError(kind)

## Step 7 — Metrics

In [ ]:
def adaptive_ece(y_true, p, n_bins=15):
    # Equal-mass (adaptive) bins.
    p = np.clip(np.asarray(p, float), 1e-9, 1 - 1e-9)
    o = np.argsort(p)
    p, y_true = p[o], np.asarray(y_true)[o]
    e = 0.0
    for b in np.array_split(np.arange(len(p)), n_bins):
        if len(b):
            e += (len(b) / len(p)) * abs(p[b].mean() - y_true[b].mean())
    return e


def score(task, y_true, raw):
    if task == "clf":
        p = 1.0 / (1.0 + np.exp(-raw))
        return dict(metric=accuracy_score(y_true, (p > 0.5).astype(int)),
                    ece=adaptive_ece(y_true, p), brier=brier_score_loss(y_true, p))
    return dict(metric=r2_score(y_true, raw), ece=np.nan, brier=np.nan)

## Step 8 — DatasetsFive of their eight. Each loader is wrapped so a failed download skips that dataset rather thankilling the run. Covertype is subsampled to `MAX_TRAIN_ROWS`; say so in your methods section.Note on calibration: breast cancer's test set is only ~114 rows, so with 15 equal-mass bins eachholds about 7 points and its ECE is noise. **Covertype and Mushroom carry the calibration section.**

In [ ]:
def _cap(Xtr, Xte, ytr, yte, seed):
    rng = np.random.RandomState(seed)
    if len(Xtr) > MAX_TRAIN_ROWS:
        i = rng.choice(len(Xtr), MAX_TRAIN_ROWS, replace=False); Xtr, ytr = Xtr[i], ytr[i]
    if len(Xte) > MAX_TEST_ROWS:
        i = rng.choice(len(Xte), MAX_TEST_ROWS, replace=False); Xte, yte = Xte[i], yte[i]
    return Xtr, Xte, ytr, yte


def _split(Xr, yr, task, seed):
    strat = yr if task == "clf" else None
    Xtr, Xte, ytr, yte = train_test_split(Xr, yr, test_size=0.2, random_state=seed, stratify=strat)
    return _cap(np.asarray(Xtr, float), np.asarray(Xte, float),
                np.asarray(ytr), np.asarray(yte), seed)


def _openml_numeric(name, version=1):
    d = fetch_openml(name, version=version, as_frame=True)
    Xdf = d.data.copy()
    cat = Xdf.select_dtypes(include=["category", "object"]).columns
    if len(cat):
        Xdf[cat] = OrdinalEncoder().fit_transform(Xdf[cat].astype(str))
    Xdf = Xdf.fillna(-1)
    yv = pd.Categorical(d.target).codes
    return Xdf.to_numpy(float), np.asarray(yv)


LOADERS = {
    "breast_cancer":      ("clf", lambda: load_breast_cancer(return_X_y=True)),
    "california_housing": ("reg", lambda: (fetch_california_housing().data,
                                           fetch_california_housing().target)),
    "covertype_binary":   ("clf", lambda: (lambda d: (d.data, (d.target == 2).astype(int)))(fetch_covtype())),
    "mushroom":           ("clf", lambda: _openml_numeric("mushroom")),
    "kr-vs-kp":           ("clf", lambda: _openml_numeric("kr-vs-kp")),
}

def cached(name, loader):
    # Download once, then reuse from Drive forever.
    p = f"{CACHE}/{name}.npz"
    if os.path.exists(p):
        z = np.load(p, allow_pickle=True)
        return z["X"], z["y"], True
    Xr, yr = loader()
    Xr, yr = np.asarray(Xr, float), np.asarray(yr)
    np.savez_compressed(p, X=Xr, y=yr)
    return Xr, yr, False


RAW = {}
for name, (task, fn) in LOADERS.items():
    try:
        t0 = time.time()
        Xr, yr, from_cache = cached(name, fn)
        RAW[name] = (task, Xr, yr)
        tag = "from cache" if from_cache else f"downloaded in {time.time()-t0:.0f}s"
        print(f"  {name:20s} {np.shape(Xr)}  task={task}  ({tag})")
    except Exception as e:
        print(f"  SKIPPED {name:20s} ({type(e).__name__}: {str(e)[:60]})")

if not RAW:
    print("\nNo datasets loaded. Falling back to bundled sets.")
    Xd, yd = load_breast_cancer(return_X_y=True); RAW["breast_cancer"] = ("clf", Xd, yd)
    d = load_diabetes(); RAW["diabetes_fallback"] = ("reg", d.data, d.target)

print(f"\n{len(RAW)} datasets ready")

## Step 9 — The joint gridThis is the change that makes the comparison legitimate. Every arm sweeps structure, ensemble sizeand bin count **together**, so that at any memory budget you can ask what the best model each armcan produce actually scores. That is their Figure 4 protocol.Each fitted model is priced four ways: their float32 accounting, their fp16 accounting, theirarray-layout accounting, and the full ToaD layout.

In [ ]:
_rows = []

def add_row(**kw):
    _rows.append(kw)


def ckpt(tag):
    return f"{CKPTDIR}/{tag}.csv"


parts = []
t_start = time.time()
for ds, (task, Xr, yr) in RAW.items():
    for seed in SEEDS:
        tag = f"lgb_{ds}_s{seed}"
        if os.path.exists(ckpt(tag)):
            parts.append(pd.read_csv(ckpt(tag)))
            print(f"  {ds} seed={seed} LightGBM: loaded from checkpoint, skipped")
            continue

        _rows.clear()
        Xtr, Xte, ytr, yte = _split(Xr, yr, task, seed)
        Model = lgb.LGBMClassifier if task == "clf" else lgb.LGBMRegressor
        nfeat = Xtr.shape[1]

        for (nl, dep) in STRUCT:
            for ne in N_EST:
                for mb in MAX_BIN:
                    m = Model(num_leaves=nl, max_depth=dep, n_estimators=ne, max_bin=mb,
                              verbose=-1, random_state=seed, n_jobs=-1).fit(Xtr, ytr)
                    r, _ = tree_roots(m.booster_)
                    raw = m.booster_.predict(Xte, raw_score=True)
                    add_row(dataset=ds, seed=seed, arm="lgb",
                            config=f"nl{nl}_d{dep}_n{ne}_b{mb}",
                            toad_kb=KB(toad_bits(r, nfeat)),
                            toadplus_kb=KB(toad_bits_plus(r, nfeat)),
                            padding=padding_waste(r),
                            f32_kb=KB(baseline_bits(r, 128)),
                            f16_kb=KB(baseline_bits(r, 64)),
                            array_kb=KB(array_layout_bits(r)),
                            **score(task, yte, raw))

                    # clustering arms only on the full-bin models, to keep the grid bounded
                    if mb == 255:
                        for kind, k in CLUSTER_CFGS:
                            rc = apply_cluster(r, kind, k)
                            lb = k if kind == "quant" else 32
                            add_row(dataset=ds, seed=seed, arm=f"lgb+{kind}",
                                    config=f"nl{nl}_d{dep}_n{ne}_{kind}{k}",
                                    toad_kb=KB(toad_bits(rc, nfeat)),
                                    toadplus_kb=KB(toad_bits_plus(rc, nfeat, leaf_bits=lb)),
                                    padding=padding_waste(rc),
                                    f32_kb=KB(baseline_bits(rc, 128)),
                                    f16_kb=KB(baseline_bits(rc, 64)),
                                    array_kb=KB(array_layout_bits(rc)),
                                    **score(task, yte, predict_vec(rc, Xte)))
        part = pd.DataFrame(_rows)
        part.to_csv(ckpt(tag), index=False)          # checkpoint immediately
        parts.append(part)
        print(f"  {ds} seed={seed} LightGBM grid done, checkpointed "
              f"({time.time()-t_start:.0f}s elapsed)")

grid = pd.concat(parts, ignore_index=True)
grid.to_csv(f"{OUTDIR}/grid_lgb.csv", index=False)
print(f"\n{len(grid)} LightGBM rows -> {OUTDIR}/grid_lgb.csv")

## Step 10 — CatBoost armsGeneral layout (apples to apples) and level-sharing oblivious layout (your contribution).

In [ ]:
from catboost import CatBoostClassifier, CatBoostRegressor

def catboost_node_count(cbj):
    # oblivious tree of depth D expands to a complete binary tree
    return sum(2 ** (len(t.get("splits", [])) + 1) - 1 for t in cbj.get("oblivious_trees", []))


def catboost_toad_bits(cbj, n_input_features, oblivious=False, meta_bits=64, leaf_bits=32):
    trees = cbj.get("oblivious_trees", [])
    feats, thr, leaves, depths = set(), {}, set(), []
    for t in trees:
        sp = t.get("splits", [])
        depths.append(len(sp))
        for s in sp:
            f = s.get("float_feature_index")
            if f is None:
                continue
            feats.add(f)
            thr.setdefault(f, set()).add(float(s["border"]))
        for lv in t.get("leaf_values", []):
            leaves.add(float(np.float32(lv)))

    nF, nL = len(feats), len(leaves)
    maxT = max((len(v) for v in thr.values()), default=1)
    fib = bits_for(n_input_features)
    gt = sum(len(thr[f]) * thr_width_and_type(thr[f])[0] for f in feats)
    mp = nF * (fib + 3 + 1 + bits_for(maxT))
    lv = nL * leaf_bits + (2 * 32 if leaf_bits < 32 else 0)
    frb, tib, lib = bits_for(nF + 1), bits_for(maxT), bits_for(nL)

    tr = 0
    for D in depths:
        if oblivious:
            tr += D * (frb + tib) + (2 ** D) * lib      # one split per LEVEL
        else:
            tr += (2 ** (D + 1) - 1) * (frb + max(tib, lib))
    return meta_bits + gt + mp + lv + tr


cb_parts = []
for ds, (task, Xr, yr) in RAW.items():
    for seed in SEEDS:
        tag = f"cb_{ds}_s{seed}"
        if os.path.exists(ckpt(tag)):
            cb_parts.append(pd.read_csv(ckpt(tag)))
            print(f"  {ds} seed={seed} CatBoost: loaded from checkpoint, skipped")
            continue
        cb_rows = []
        Xtr, Xte, ytr, yte = _split(Xr, yr, task, seed)
        CB = CatBoostClassifier if task == "clf" else CatBoostRegressor
        nfeat = Xtr.shape[1]
        for dep in CB_DEPTH:
            for it in CB_ITERS:
                m = CB(depth=dep, iterations=it, verbose=0, random_seed=seed,
                       allow_writing_files=False).fit(Xtr, ytr)
                path = f"{CKPTDIR}/_cb_tmp.json"
                m.save_model(path, format="json")
                cbj = json.load(open(path))
                if task == "clf":
                    p = m.predict_proba(Xte)[:, 1]
                    s = dict(metric=accuracy_score(yte, (p > 0.5).astype(int)),
                             ece=adaptive_ece(yte, p), brier=brier_score_loss(yte, p))
                else:
                    s = dict(metric=r2_score(yte, m.predict(Xte)), ece=np.nan, brier=np.nan)
                nn = catboost_node_count(cbj)
                for arm, obl in [("catboost", False), ("catboost_oblivious", True)]:
                    cb_rows.append(dict(dataset=ds, seed=seed, arm=arm, config=f"d{dep}_i{it}",
                                        toad_kb=KB(catboost_toad_bits(cbj, nfeat, obl)),
                                        toadplus_kb=KB(catboost_toad_bits(cbj, nfeat, obl, leaf_bits=8)),
                                        padding=1.0,
                                        f32_kb=KB(nn * 128), f16_kb=KB(nn * 64),
                                        array_kb=KB(nn * 64), **s))
        cbp = pd.DataFrame(cb_rows)
        cbp.to_csv(ckpt(tag), index=False)
        cb_parts.append(cbp)
        print(f"  {ds} seed={seed} CatBoost done, checkpointed "
              f"({time.time()-t_start:.0f}s elapsed)")

res = pd.concat([grid] + cb_parts, ignore_index=True)
res.to_csv(f"{OUTDIR}/all_results.csv", index=False)
print(f"\n{len(res)} total rows -> {OUTDIR}/all_results.csv")
print("Safe to close the tab now - everything is on Drive.")

## Step 11 — Accuracy-memory envelopes

At each memory budget, the best quality each curve reaches. This is the vertical read, and it is
their Figure 4.

The *horizontal* read (memory needed to hit a fixed quality, which is what a compression ratio
actually is) is deliberately not done here. Against a single shared baseline it mixes the learner
difference into the compression number. Part 2 does it correctly, per learner.

In [ ]:
CURVES = [
    ("LightGBM (fp32 baseline)",        lambda d: d[d.arm == "lgb"],                       "f32_kb"),
    ("LightGBM (fp16 baseline)",        lambda d: d[d.arm == "lgb"],                       "f16_kb"),
    ("LightGBM (array layout)",         lambda d: d[d.arm == "lgb"],                       "array_kb"),
    ("LightGBM (ToaD layout)",          lambda d: d[d.arm == "lgb"],                       "toad_kb"),
    ("LightGBM + leaf cluster (ToaD)",  lambda d: d[d.arm == "lgb+leaf"],                  "toad_kb"),
    ("LightGBM + both cluster (ToaD)",  lambda d: d[d.arm.isin(["lgb+both","lgb+thr"])],   "toad_kb"),
    ("CatBoost (ToaD layout)",          lambda d: d[d.arm == "catboost"],                  "toad_kb"),
    ("CatBoost (oblivious layout)",     lambda d: d[d.arm == "catboost_oblivious"],        "toad_kb"),
    ("LightGBM (ToaD+ layout)",         lambda d: d[d.arm == "lgb"],                       "toadplus_kb"),
    ("LightGBM + quant leaves (ToaD+)", lambda d: d[d.arm == "lgb+quant"],                 "toadplus_kb"),
    ("LightGBM + leaf cl. (ToaD+)",     lambda d: d[d.arm == "lgb+leaf"],                  "toadplus_kb"),
    ("CatBoost obliv. (ToaD+)",         lambda d: d[d.arm == "catboost_oblivious"],        "toadplus_kb"),
]

BUDGETS = np.array([0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 128])


def vertical_envelope(df, mem_col):
    # Average across seeds first, so a lucky seed cannot define the envelope.
    g = df.groupby("config", as_index=False).agg({mem_col: "mean", "metric": "mean"})
    return [g.loc[g[mem_col] <= b, "metric"].max() if (g[mem_col] <= b).any() else np.nan
            for b in BUDGETS]


def min_mem_for(df, mem_col, level):
    g = df.groupby("config", as_index=False).agg({mem_col: "mean", "metric": "mean"})
    s = g[g.metric >= level]
    return s[mem_col].min() if len(s) else np.nan


env_rows, ratio_rows = [], []
for ds in res.dataset.unique():
    d = res[res.dataset == ds]
    for label, sel, mem in CURVES:
        sub = sel(d)
        if not len(sub):
            continue
        env_rows.append(dict(dataset=ds, curve=label,
                             **{f"{b}KB": v for b, v in zip(BUDGETS, vertical_envelope(sub, mem))}))


env = pd.DataFrame(env_rows)
env.to_csv(f"{OUTDIR}/envelopes.csv", index=False)
print(f"vertical envelopes computed for {env.dataset.nunique()} datasets")
print("(compression RATIOS come later, in Part 2, with correct per-learner denominators)")

## Step 12 — Plots

In [ ]:
dss = list(res.dataset.unique())
fig, axes = plt.subplots(1, len(dss), figsize=(5.6 * len(dss), 4.6), squeeze=False)

STYLE = {
    "LightGBM (fp32 baseline)":       ("o", "#999999", "-"),
    "LightGBM (fp16 baseline)":       ("v", "#bbbbbb", "-"),
    "LightGBM (array layout)":        ("<", "#7f7f7f", ":"),
    "LightGBM (ToaD layout)":         ("s", "#1f77b4", "-"),
    "LightGBM + leaf cluster (ToaD)": ("^", "#d62728", "-"),
    "LightGBM + both cluster (ToaD)": ("P", "#e377c2", "--"),
    "CatBoost (ToaD layout)":         ("D", "#9467bd", "--"),
    "CatBoost (oblivious layout)":    ("*", "#ff7f0e", "-"),
    "LightGBM (ToaD+ layout)": ("h", "#17becf", "-"),
}

for ax, ds in zip(axes[0], dss):
    e = env[env.dataset == ds]
    for _, r in e.iterrows():
        vals = [r[f"{b}KB"] for b in BUDGETS]
        mk, c, ls = STYLE.get(r.curve, ("o", "gray", "-"))
        ax.plot(BUDGETS, vals, marker=mk, color=c, linestyle=ls, label=r.curve,
                markersize=7, alpha=0.9)
    ax.set_xscale("log")
    ax.set_xlabel("memory budget (KB, log)")
    task = RAW[ds][0] if ds in RAW else "clf"
    ax.set_ylabel("accuracy" if task == "clf" else "R squared")
    ax.set_title(ds)
    ax.grid(alpha=0.3)
axes[0][0].legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.savefig(f"{FIGDIR}/envelopes.png", dpi=150)
plt.show()

# leaf vs threshold clustering, head to head
sub = res[res.arm.isin(["lgb", "lgb+leaf", "lgb+thr", "lgb+both"])]
fig, axes = plt.subplots(1, len(dss), figsize=(5.2 * len(dss), 4.2), squeeze=False)
for ax, ds in zip(axes[0], dss):
    d = sub[sub.dataset == ds]
    for arm, g in d.groupby("arm"):
        g = g.groupby("config", as_index=False).agg({"toad_kb": "mean", "metric": "mean"})
        g = g.sort_values("toad_kb")
        ax.scatter(g.toad_kb, g.metric, s=22, alpha=0.65, label=arm)
    ax.set_xscale("log"); ax.set_xlabel("ToaD-layout KB (log)"); ax.set_title(ds)
    ax.grid(alpha=0.3)
axes[0][0].set_ylabel("quality"); axes[0][0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/clustering_arms.png", dpi=150); plt.show()

## Step 13 — Calibration (only where the test set is large enough)

In [ ]:
clf_ds = [d for d in dss if RAW.get(d, ("reg",))[0] == "clf"]
cal = res[(res.dataset.isin(clf_ds)) & (res.ece.notna())].copy()

# a small test set makes ECE meaningless; flag rather than silently report
print("test-set sizes (ECE is unreliable below ~2000 rows):")
for ds in clf_ds:
    task, Xr, yr = RAW[ds]
    _, Xte, _, _ = _split(Xr, yr, task, SEEDS[0])
    print(f"  {ds:20s} {len(Xte):6d} rows   {'OK' if len(Xte) >= 2000 else 'TOO SMALL - do not quote'}")

if len(cal):
    piv = (cal.groupby(["dataset", "arm"])
              .agg(mean_ece=("ece", "mean"), mean_brier=("brier", "mean"),
                   mean_metric=("metric", "mean"), mean_kb=("toad_kb", "mean"))
              .round(4))
    print("\n", piv.to_string())

## Step 14 — ToaD versus ToaD+ head to headSame models, two accountings. Any gap here is layout only, with predictions unchanged.

In [ ]:
lg = res[res.arm == "lgb"].copy()
lg["gain"] = lg.toad_kb / lg.toadplus_kb


def _depth_of(cfg):
    for part in str(cfg).split("_"):
        if part.startswith("d") and (part[1:].isdigit() or part[1:] == "-1"):
            return int(part[1:])
    return np.nan


lg["depth"] = lg.config.map(_depth_of)

print("lossless layout gain (ToaD -> ToaD+), LightGBM models:\n")
print(lg.groupby("dataset")
        .agg(mean_padding=("padding", "mean"), mean_gain=("gain", "mean"),
             max_gain=("gain", "max")).round(2).to_string())

print("\n\ngain vs tree depth (this is the padding effect):\n")
print(lg.dropna(subset=["depth"]).groupby("depth")[["padding", "gain"]]
        .mean().round(2).to_string())

q = res[res.arm == "lgb+quant"].copy()
if len(q):
    q["bits"] = q.config.map(lambda c: int(str(c).rsplit("quant", 1)[-1]))
    print("\n\nleaf quantization (LOSSY - read the acc column, not just kb):\n")
    print(q.groupby(["dataset", "bits"])
            .agg(kb=("toadplus_kb", "mean"), acc=("metric", "mean")).round(4).to_string())
    print("\nunquantized mean accuracy for reference:\n",
          res[res.arm == "lgb"].groupby("dataset").metric.mean().round(4).to_string())

# Part 2 — Corrected ratios and the missing analyses

Part 1 measures models. Part 2 interprets them, and fixes four things a naive reading gets wrong.

1. **Ratios need per-learner denominators.** Pricing CatBoost against LightGBM's float32 baseline
   folds "CatBoost is the better learner here" into the compression number, inflating it where
   CatBoost wins and deflating it where it loses. Step 12 prices each learner against itself and
   reports the cross-learner comparison separately, labelled as model selection.
2. **Accuracy-optimal models are shallow**, so a succinct layout has little padding to remove on
   the frontier. Step 13 measures that and scopes the claim.
3. **`max_depth` was capped everywhere**, yet LightGBM's default is `max_depth=-1`. Step 14 adds
   that regime, where complete-tree indexing should hurt most.
4. **Calibration was grid-averaged** and three of four test sets were too small. Steps 15 and 16
   fix both.

## Step 12 — Compression ratios, per learner

Quality levels come from the range both learners can actually reach, so no ratio is computed at a level one of them cannot hit.

In [ ]:
LGB_ARMS = ["lgb", "lgb+leaf", "lgb+thr", "lgb+both", "lgb+quant"]
CB_ARMS  = ["catboost", "catboost_oblivious"]


def min_mem_for(df, mem_col, level):
    if not len(df):
        return np.nan
    g = df.groupby("config", as_index=False).agg({mem_col: "mean", "metric": "mean"})
    s = g[g.metric >= level]
    return s[mem_col].min() if len(s) else np.nan


WITHIN = [
    ("fp16 baseline",            "lgb",        "f16_kb"),
    ("array layout",             "lgb",        "array_kb"),
    ("ToaD layout",              "lgb",        "toad_kb"),
    ("ToaD+ layout (lossless)",  "lgb",        "toadplus_kb"),
    ("ToaD + leaf cluster",      "lgb+leaf",   "toad_kb"),
    ("ToaD+ & leaf cluster",     "lgb+leaf",   "toadplus_kb"),
    ("ToaD+ & quant leaves",     "lgb+quant",  "toadplus_kb"),
    ("ToaD + both cluster",      "lgb+both",   "toad_kb"),
]
WITHIN_CB = [
    ("fp16 baseline",            "catboost",           "f16_kb"),
    ("ToaD layout",              "catboost",           "toad_kb"),
    ("oblivious layout",         "catboost_oblivious", "toad_kb"),
    ("oblivious + ToaD+",        "catboost_oblivious", "toadplus_kb"),
]

within_rows, cross_rows = [], []
for ds in res.dataset.unique():
    d = res[res.dataset == ds]
    L, C = d[d.arm == "lgb"], d[d.arm == "catboost"]
    if not len(L) or not len(C):
        continue
    lo = max(L.metric.quantile(0.25), C.metric.quantile(0.25))
    hi = min(L.metric.max(), C.metric.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        continue
    levels = np.linspace(lo, hi * 0.999, 5)

    for lvl in levels:
        ref_l = min_mem_for(L, "f32_kb", lvl)
        for label, arm, mem in WITHIN:
            m = min_mem_for(d[d.arm == arm], mem, lvl)
            within_rows.append(dict(dataset=ds, learner="LightGBM", technique=label, level=lvl,
                                    ratio=ref_l / m if m and np.isfinite(m) and m > 0 else np.nan))
        ref_c = min_mem_for(C, "f32_kb", lvl)
        for label, arm, mem in WITHIN_CB:
            m = min_mem_for(d[d.arm == arm], mem, lvl)
            within_rows.append(dict(dataset=ds, learner="CatBoost", technique=label, level=lvl,
                                    ratio=ref_c / m if m and np.isfinite(m) and m > 0 else np.nan))
        # cross-learner, best storage for each
        bl = min([min_mem_for(d[d.arm == a], m, lvl) for a, m in
                  [("lgb", "toadplus_kb"), ("lgb+leaf", "toadplus_kb"), ("lgb+quant", "toadplus_kb")]]
                 or [np.nan])
        bc = min_mem_for(d[d.arm == "catboost_oblivious"], "toadplus_kb", lvl)
        cross_rows.append(dict(dataset=ds, level=lvl, best_lgb_kb=bl, best_cb_kb=bc,
                               cb_over_lgb=bl / bc if bc and np.isfinite(bc) and bc > 0 else np.nan))

within = pd.DataFrame(within_rows)
cross = pd.DataFrame(cross_rows)

print("=== WITHIN-LEARNER compression (vs that learner's own fp32) ===\n")
for learner in ["LightGBM", "CatBoost"]:
    w = within[within.learner == learner]
    if not len(w):
        continue
    piv = w.pivot_table(index="technique", columns="dataset", values="ratio", aggfunc="mean")
    order = [t for _, t in sorted({r[0]: r[0] for r in [(x[0],) for x in
             (WITHIN if learner == "LightGBM" else WITHIN_CB)]}.items())]
    print(f"--- {learner} ---")
    print(piv.round(2).to_string(), "\n")

print("\n=== CROSS-LEARNER (model selection, NOT compression) ===")
print("how many times smaller the best CatBoost model is than the best LightGBM model, same quality\n")
print(cross.groupby("dataset").cb_over_lgb.mean().round(2).to_string())

within.to_csv(f"{OUTDIR}/within_learner_ratios.csv", index=False)
cross.to_csv(f"{OUTDIR}/cross_learner_ratios.csv", index=False)

## Step 13 — Why the succinct layout looks weak on the frontier

A succinct encoding removes padding, so it only helps models that have padding. If the accuracy-optimal model at every budget is shallow and balanced, there is nothing to remove. This decides whether ToaD+ is a headline or a paragraph.

In [ ]:
def depth_of(cfg):
    for p in str(cfg).split("_"):
        if p.startswith("d") and (p[1:].isdigit() or p[1:] == "-1"):
            return int(p[1:])
    return np.nan


BUDGETS = np.array([0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 128])
rows = []
for ds in res.dataset.unique():
    L = res[(res.dataset == ds) & (res.arm == "lgb")].copy()
    L["depth"] = L.config.map(depth_of)
    for b in BUDGETS:
        s = L[L.toad_kb <= b]
        if not len(s):
            continue
        i = s.metric.idxmax()
        rows.append(dict(dataset=ds, budget=b, depth=L.loc[i, "depth"],
                         padding=L.loc[i, "padding"], metric=L.loc[i, "metric"],
                         gain=L.loc[i, "toad_kb"] / L.loc[i, "toadplus_kb"]))
fr = pd.DataFrame(rows)

print("frontier-selected models (the ones on the accuracy-memory curve):\n")
print(fr.groupby("dataset").agg(mean_depth=("depth", "mean"),
                                mean_padding=("padding", "mean"),
                                mean_toadplus_gain=("gain", "mean")).round(2).to_string())

allm = res[res.arm == "lgb"]
print("\nfor comparison, ALL models in the grid:\n")
print(allm.groupby("dataset").agg(mean_padding=("padding", "mean"),
                                  mean_gain=("toad_kb", "mean")).round(2).to_string())
print("\n(if frontier padding is ~1.0 while grid padding is higher, the ToaD+ gain is real but")
print(" lives off the accuracy-optimal frontier - say exactly that in the paper)")
fr.to_csv(f"{OUTDIR}/frontier_composition.csv", index=False)

## Step 14 — Unlimited-depth configurations

Every configuration in Part 1 capped `max_depth`. LightGBM's default is `max_depth=-1`, where leaf-wise growth produces strongly unbalanced trees. Checkpointed like the main grid.

In [ ]:
UNLIM_LEAVES = [31, 63, 127, 255, 511]      # max_depth = -1 throughout
UNLIM_NEST   = [30, 60, 120]

unlim_parts = []
for ds, (task, Xr, yr) in RAW.items():
    for seed in SEEDS:
        tag = f"unlim_{ds}_s{seed}"
        if os.path.exists(ckpt(tag)):
            unlim_parts.append(pd.read_csv(ckpt(tag)))
            print(f"  {ds} seed={seed} unlimited-depth: loaded from checkpoint, skipped")
            continue

        rows_u = []
        Xtr, Xte, ytr, yte = _split(Xr, yr, task, seed)
        Model = lgb.LGBMClassifier if task == "clf" else lgb.LGBMRegressor
        nfeat = Xtr.shape[1]
        for nl in UNLIM_LEAVES:
            for ne in UNLIM_NEST:
                m = Model(num_leaves=nl, max_depth=-1, n_estimators=ne, max_bin=255,
                          verbose=-1, random_state=seed, n_jobs=-1).fit(Xtr, ytr)
                r, _ = tree_roots(m.booster_)
                raw = m.booster_.predict(Xte, raw_score=True)
                met = (accuracy_score(yte, (1/(1+np.exp(-raw)) > 0.5).astype(int))
                       if task == "clf" else r2_score(yte, raw))
                rows_u.append(dict(dataset=ds, seed=seed, arm="lgb_unlimited",
                                   config=f"nl{nl}_d-1_n{ne}",
                                   toad_kb=KB(toad_bits(r, nfeat)),
                                   toadplus_kb=KB(toad_bits_plus(r, nfeat)),
                                   padding=padding_waste(r),
                                   f32_kb=KB(baseline_bits(r, 128)),
                                   f16_kb=KB(baseline_bits(r, 64)),
                                   array_kb=KB(array_layout_bits(r)), metric=met,
                                   mean_depth=np.mean([tree_depth(t) for t in r])))
        part = pd.DataFrame(rows_u)
        part.to_csv(ckpt(tag), index=False)
        unlim_parts.append(part)
        print(f"  {ds} seed={seed} unlimited-depth grid done, checkpointed")

U = pd.concat(unlim_parts, ignore_index=True)
U["gain"] = U.toad_kb / U.toadplus_kb
U.to_csv(f"{OUTDIR}/unlimited_depth.csv", index=False)

print("\npadding and ToaD+ gain with max_depth unlimited (LightGBM's own default):\n")
print(U.groupby("dataset").agg(mean_tree_depth=("mean_depth", "mean"),
                               mean_padding=("padding", "mean"),
                               mean_gain=("gain", "mean"),
                               max_gain=("gain", "max")).round(2).to_string())

print("\naccuracy check - are these models any good?\n")
cap = res[res.arm == "lgb"].groupby("dataset").metric.max().round(4)
unl = U.groupby("dataset").metric.max().round(4)
print(pd.DataFrame({"best_capped_depth": cap, "best_unlimited": unl}).to_string())
print("\n(if unlimited-depth models are clearly worse, the padding result is about a configuration")
print(" people actually use, not the accuracy-optimal frontier - state which one in the paper)")

## Step 15 — Calibration at matched memory budget

Part 1's table averaged ECE over every model in the grid, so it partly measured which sizes each arm contained. Here the best-accuracy model within each budget is selected and *its* calibration reported, the same rule as the accuracy envelope.

In [ ]:
CAL_ARMS = ["lgb", "lgb+leaf", "lgb+quant", "lgb+both", "catboost", "catboost_oblivious"]
rows = []
for ds in res.dataset.unique():
    d = res[(res.dataset == ds) & res.ece.notna()]
    if not len(d):
        continue
    for arm in CAL_ARMS:
        a = d[d.arm == arm]
        if not len(a):
            continue
        mem = "toad_kb"
        for b in BUDGETS:
            s = a[a[mem] <= b]
            if not len(s):
                continue
            i = s.metric.idxmax()
            rows.append(dict(dataset=ds, arm=arm, budget=b, kb=a.loc[i, mem],
                             metric=a.loc[i, "metric"], ece=a.loc[i, "ece"],
                             brier=a.loc[i, "brier"]))
cal = pd.DataFrame(rows)
if len(cal):
    cal.to_csv(f"{OUTDIR}/calibration_matched_budget.csv", index=False)
    for ds in cal.dataset.unique():
        print(f"--- {ds} : ECE of the best model within each budget ---")
        print(cal[cal.dataset == ds].pivot_table(index="arm", columns="budget",
                                                 values="ece").round(4).to_string(), "\n")

## Step 16 — Pooled out-of-fold calibration

Mushroom has 8124 rows and kr-vs-kp 3196, so a 20% test split leaves too few points for a 15-bin ECE, and resampling cannot fix that. Repeated stratified CV with pooled out-of-fold predictions uses every row. Only a few representative configurations are run. Checkpointed per dataset.

In [ ]:
CV_CONFIGS = [("small", dict(num_leaves=8, max_depth=3, n_estimators=30)),
              ("medium", dict(num_leaves=16, max_depth=4, n_estimators=60)),
              ("large", dict(num_leaves=32, max_depth=5, n_estimators=120))]
N_SPLITS, N_REPEATS = 5, 3


from catboost import CatBoostClassifier

oof_parts = []
for ds, (task, Xr, yr) in RAW.items():
    if task != "clf":
        continue
    tag = f"oof_{ds}"
    if os.path.exists(ckpt(tag)):
        oof_parts.append(pd.read_csv(ckpt(tag)))
        print(f"  {ds} pooled CV: loaded from checkpoint, skipped")
        continue
    oof_rows = []
    X, y = np.asarray(Xr, float), np.asarray(yr)
    if len(X) > MAX_TRAIN_ROWS:
        i = np.random.RandomState(1).choice(len(X), MAX_TRAIN_ROWS, replace=False)
        X, y = X[i], y[i]
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=1)
    for name, params in CV_CONFIGS:
        preds = {"lgb": [], "catboost": []}
        truth = []
        for tr, te in cv.split(X, y):
            ml = lgb.LGBMClassifier(**params, max_bin=255, verbose=-1,
                                    random_state=1, n_jobs=-1).fit(X[tr], y[tr])
            preds["lgb"].append(ml.predict_proba(X[te])[:, 1])
            mc = CatBoostClassifier(depth=params["max_depth"], iterations=params["n_estimators"],
                                    verbose=0, random_seed=1,
                                    allow_writing_files=False).fit(X[tr], y[tr])
            preds["catboost"].append(mc.predict_proba(X[te])[:, 1])
            truth.append(y[te])
        yy = np.concatenate(truth)
        for arm, pl in preds.items():
            pp = np.concatenate(pl)
            oof_rows.append(dict(dataset=ds, size=name, arm=arm, n_pooled=len(yy),
                                 ece=adaptive_ece(yy, pp), brier=brier_score_loss(yy, pp),
                                 acc=accuracy_score(yy, (pp > 0.5).astype(int))))
        print(f"  {ds} / {name} done ({len(yy)} pooled predictions)")
    _p = pd.DataFrame(oof_rows)
    _p.to_csv(ckpt(tag), index=False)
    oof_parts.append(_p)

oof = pd.concat(oof_parts, ignore_index=True) if oof_parts else pd.DataFrame()
if len(oof):
    oof.to_csv(f"{OUTDIR}/oof_calibration.csv", index=False)
    print("\npooled out-of-fold calibration (every row contributes):\n")
    print(oof.pivot_table(index=["dataset", "size"], columns="arm",
                          values=["ece", "acc"]).round(4).to_string())

## Step 17 — Paper-ready summary

In [ ]:
print("FILES WRITTEN\n")
for f in ["within_learner_ratios.csv", "cross_learner_ratios.csv", "frontier_composition.csv",
          "unlimited_depth.csv", "calibration_matched_budget.csv", "oof_calibration.csv"]:
    p = f"{OUTDIR}/{f}"
    print(f"  {'OK ' if os.path.exists(p) else '-- '} {f}")

print("\n\nHEADLINE NUMBERS TO CHECK BEFORE WRITING\n")
if len(within):
    w = within[within.learner == "LightGBM"].pivot_table(index="technique", columns="dataset",
                                                         values="ratio", aggfunc="mean")
    print("1. LightGBM within-learner compression (this is the honest compression claim):")
    print(w.round(2).to_string())
if len(cross):
    print("\n2. CatBoost advantage at matched quality (LABEL AS MODEL SELECTION, not compression):")
    print(cross.groupby("dataset").cb_over_lgb.mean().round(2).to_string())
try:
    print("\n3. ToaD+ gain, capped depth vs unlimited depth:")
    print("   capped   :", round(float(fr.gain.mean()), 2), "x (frontier models)")
    print("   unlimited:", round(float(U.gain.mean()), 2), "x  max", round(float(U.gain.max()), 2), "x")
except Exception:
    pass

# Reading the results

**1. Did validation pass?** Step 5 must print `VALIDATION PASSED`. Nothing below means anything otherwise.

**2. The honest compression claim** is the LightGBM table in Step 12, priced against LightGBM's own
float32 baseline. The `ToaD layout` row is what their memory layout buys with training completely
unmodified. Their headline is 4-16x for layout plus penalties together, so the difference is what
the penalties contributed.

**3. Leaf versus threshold clustering.** In `figures/clustering_arms.png`, compare `lgb+leaf` against
`lgb+thr`. Leaf clustering has been the far stronger lever; threshold clustering was close to a no-op
on the categorical datasets, where ordinal-encoded features already have fewer distinct thresholds
than the cluster count. That mechanism deserves a paragraph, because their threshold penalty has
nothing to bite on in that regime.

**4. Does the succinct layout earn a headline?** Compare Step 13 (frontier, capped depth) against
Step 14 (unlimited depth). If frontier padding sits near 1.0 while unlimited-depth padding is orders
of magnitude higher, the finding is *fragility in the default configuration*, not "we compressed
further". Write it that way, and check Step 14's accuracy comparison before leaning on it.

**5. Cross-learner numbers are model selection, not compression.** Label them so. The sign flips
between datasets, which is worth reporting in itself.

**6. Calibration.** Use Step 15 (matched budget) and Step 16 (pooled CV). Ignore Part 1's
grid-averaged table. Mushroom is saturated: report it, do not lean on it.

# Before you write

- **Verify assumptions A1-A3 against their repo.** Clone `TinyAIoT/LightGBM-ToaD` at v1.0.0, find
  their memory computation, reconcile against `toad_bits`. If yours disagrees on the same model, fix
  yours and document it. Highest-risk item in the project.
- **Reproduce one published point.** They report binary Covertype at 2 KB / 69% for ToaD, matched by
  quantized LightGBM only at 8 KB and float32 LightGBM at 16 KB. Your fp32 and fp16 curves should
  land near those.
- **Check whether they cap depth throughout.** If they do, the unlimited-depth result is a scope
  condition on their method rather than an error, and should be written that way.
- **Set `SEEDS = list(range(1, 13))`** and re-run. Checkpoints mean only new seeds are computed.

# If a session drops

Re-run from the top. Cached datasets load from Drive, finished grids load from checkpoints, only
unfinished work is recomputed. To force a recompute, delete that file from
`toad_project/checkpoints/`.